In [ ]:
# In[1]:

import pandas as pd

# Load CSVs
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_df = pd.read_csv("error_logs.csv")

# Parse timestamps to UTC datetimes (Unix seconds -> UTC)
metric_df["ts"] = pd.to_datetime(metric_df["timestamp"], unit="s", utc=True)
trace_df["ts"] = pd.to_datetime(trace_df["timestamp"], unit="s", utc=True)
log_df["ts"] = pd.to_datetime(log_df["timestamp"], unit="s", utc=True)
error_df["ts"] = pd.to_datetime(error_df["timestamp"], unit="s", utc=True)

# Summaries: total rows and min/max timestamp per file
file_summaries = pd.DataFrame([
    {
        "file": "metric.csv",
        "rows": metric_df.shape[0],
        "min_ts": metric_df["ts"].min(),
        "max_ts": metric_df["ts"].max()
    },
    {
        "file": "trace.csv",
        "rows": trace_df.shape[0],
        "min_ts": trace_df["ts"].min(),
        "max_ts": trace_df["ts"].max()
    },
    {
        "file": "log.csv",
        "rows": log_df.shape[0],
        "min_ts": log_df["ts"].min(),
        "max_ts": log_df["ts"].max()
    },
    {
        "file": "error_logs.csv",
        "rows": error_df.shape[0],
        "min_ts": error_df["ts"].min(),
        "max_ts": error_df["ts"].max()
    }
])

# 1) metric.csv aggregates: unique (cmdb_id, kpi_name) with counts, top 20
metric_agg = (
    metric_df
    .groupby(["cmdb_id", "kpi_name"], as_index=False)
    .agg(count=("timestamp", "size"),
         earliest_ts=("ts", "min"),
         latest_ts=("ts", "max"))
    .sort_values("count", ascending=False)
    .head(20)
)

# 2) trace.csv aggregates: unique (cmdb_id, trace_name) with counts, top 20
trace_agg = (
    trace_df
    .groupby(["cmdb_id", "trace_name"], as_index=False)
    .agg(count=("timestamp", "size"),
         earliest_ts=("ts", "min"),
         latest_ts=("ts", "max"))
    .sort_values("count", ascending=False)
    .head(20)
)

# 3) log.csv aggregates: unique (cmdb_id, log_name) with counts, top 20
log_agg = (
    log_df
    .groupby(["cmdb_id", "log_name"], as_index=False)
    .agg(count=("timestamp", "size"),
         earliest_ts=("ts", "min"),
         latest_ts=("ts", "max"))
    .sort_values("count", ascending=False)
    .head(20)
)

# 4) error_logs.csv: counts per cmdb_id (sorted desc) and up to 5 sample messages for top cmdb_id
error_counts = (
    error_df
    .groupby("cmdb_id", as_index=False)
    .agg(count=("timestamp", "size"),
         earliest_ts=("ts", "min"),
         latest_ts=("ts", "max"))
    .sort_values("count", ascending=False)
)

# Samples for top cmdb_id (up to 5)
if not error_counts.empty:
    top_cmdb = error_counts.iloc[0]["cmdb_id"]
    error_samples_top = (
        error_df[error_df["cmdb_id"] == top_cmdb]
        .loc[:, ["ts", "cmdb_id", "message"]]
        .sort_values("ts")
        .head(5)
        .reset_index(drop=True)
    )
else:
    top_cmdb = None
    error_samples_top = pd.DataFrame(columns=["ts", "cmdb_id", "message"])

# Compact outputs (do not show full raw data)
file_summaries, metric_agg, trace_agg, log_agg, error_counts.head(20), error_samples_top

```
Out[1]:
```
summary = (
    "Summary of telemetry files:\n"
    "- metric.csv: 8,946 rows; timestamps from 2024-01-25 11:32:00 UTC to 2024-01-25 11:56:00 UTC.\n"
    "  Top metric (cmdb_id, kpi_name) pairs (examples, counts): many pairs have 25 rows each. Notable examples: "
    "ts-admin-basic-info-service (cpu, diskio, error, latency-50, latency-90), ts-travel-service and ts-travel2-service (mem, diskio, cpu, socket, latency-*), ts-voucher-service (socket).\n"
    "- trace.csv: 18,000 rows; timestamps from 2024-01-25 11:32:00 UTC to 2024-01-25 11:56:00 UTC.\n"
    "  Top trace (cmdb_id, trace_name) pairs: many traces for ts-travel2-service and ts-travel-service (e.g., trace.from_ts-seat-service.*, trace.from_root.*, trace.to_ts-travel-service.*) each with 25 rows.\n"
    "- log.csv: 2,182 rows; timestamps from 2024-01-25 11:32:00 UTC to 2024-01-25 11:56:00 UTC.\n"
    "  Top log (cmdb_id, log_name) pairs: log.error_count and log.row_count for many services (ts-admin-basic-info-service, ts-auth-service, ts-travel-service, ts-travel2-service, ts-basic-service, etc.), typically 25 rows each.\n"
    "- error_logs.csv: 0 rows (no error log records present).\n\n"
    "Overall: all telemetry files (metric, trace, log) cover the same 25-minute window and most reported metric/trace/log pairs have 25 samples each. There are no error log entries to inspect."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(             file   rows                    min_ts                    max_ts
0      metric.csv   8946 2024-01-25 11:32:00+00:00 2024-01-25 11:56:00+00:00
1       trace.csv  18000 2024-01-25 11:32:00+00:00 2024-01-25 11:56:00+00:00
2         log.csv   2182 2024-01-25 11:32:00+00:00 2024-01-25 11:56:00+00:00
3  error_logs.csv      0                       NaT                       NaT,                          cmdb_id    kpi_name  count               earliest_ts                 latest_ts
357           ts-voucher-service      socket     25 2024-01-25 11:32:00+00:00 2024-01-25 11:56:00+00:00
0    ts-admin-basic-info-service         cpu     25 2024-01-25 11:32:00+00:00 2024-01-25 11:56:00+00:00
1    ts-admin-basic-info-service      diskio     25 2024-01-25 11:32:00+00:00 2024-01-25 11:56:00+00:00
2    ts-admin-basic-info-service       error     25 2024-01-25 11:32:00+00:00 2024-01-25 11:56:00+00:00
3    ts-admin-basic-info-service  latency-50     25 2024-01-25 11:32:00+00:00 2024-01-25 11:56:00+00:00
4    ts-admin-basic-info-service  latency-90     25 2024-01-25 11:32:00+00:00 2024-01-25 11:56:00+00:00
318             ts-travel2-mongo         mem     25 2024-01-25 11:32:00+00:00 2024-01-25 11:56:00+00:00
319             ts-travel2-mongo      socket     25 2024-01-25 11:32:00+00:00 2024-01-25 11:56:00+00:00
320           ts-travel2-service         cpu     25 2024-01-25 11:32:00+00:00 2024-01-25 11:56:00+00:00
321           ts-travel2-service      diskio     25 2024-01-25 11:32:00+00:00 2024-01-25 11:56:00+00:00
322           ts-travel2-service  latency-50     25 2024-01-25 11:32:00+00:00 2024-01-25 11:56:00+00:00
323           ts-travel2-service  latency-90     25 2024-01-25 11:32:00+00:00 2024-01-25 11:56:00+00:00
324           ts-travel2-service         mem     25 2024-01-25 11:32:00+00:00 2024-01-25 11:56:00+00:00
325           ts-travel2-service      socket     25 2024-01-25 11:32:00+00:00 2024-01-25 11:56:00+00:00
310            ts-travel-service      diskio     25 2024-01-25 11:32:00+00:00 2024-01-25 11:56:00+00:00
311            ts-travel-service  latency-50     25 2024-01-25 11:32:00+00:00 2024-01-25 11:56:00+00:00
312            ts-travel-service  latency-90     25 2024-01-25 11:32:00+00:00 2024-01-25 11:56:00+00:00
313            ts-travel-service         mem     25 2024-01-25 11:32:00+00:00 2024-01-25 11:56:00+00:00
314            ts-travel-service      socket     25 2024-01-25 11:32:00+00:00 2024-01-25 11:56:00+00:00
315            ts-travel-service    workload     25 2024-01-25 11:32:00+00:00 2024-01-25 11:56:00+00:00,                 cmdb_id                                        trace_name  count               earliest_ts                 latest_ts
728  ts-travel2-service          trace.from_ts-seat-service.duration_mean     25 2024-01-25 11:32:00+00:00 2024-01-25 11:56:00+00:00
729  ts-travel2-service           trace.from_ts-seat-service.duration_p95     25 2024-01-25 11:32:00+00:00 2024-01-25 11:56:00+00:00
730  ts-travel2-service             trace.from_ts-seat-service.error_rate     25 2024-01-25 11:32:00+00:00 2024-01-25 11:56:00+00:00
731  ts-travel2-service              trace.from_ts-seat-service.row_count     25 2024-01-25 11:32:00+00:00 2024-01-25 11:56:00+00:00
732  ts-travel2-service       trace.from_ts-travel2-service.duration_mean     25 2024-01-25 11:32:00+00:00 2024-01-25 11:56:00+00:00
733  ts-travel2-service        trace.from_ts-travel2-service.duration_p95     25 2024-01-25 11:32:00+00:00 2024-01-25 11:56:00+00:00
734  ts-travel2-service          trace.from_ts-travel2-service.error_rate     25 2024-01-25 11:32:00+00:00 2024-01-25 11:56:00+00:00
735  ts-travel2-service           trace.from_ts-travel2-service.row_count     25 2024-01-25 11:32:00+00:00 2024-01-25 11:56:00+00:00
720  ts-travel2-service  trace.from_ts-admin-travel-service.duration_mean     25 2024-01-25 11:32:00+00:00 2024-01-25 11:56:00+00:00
721  ts-travel2-service   trace.from_ts-admin-travel-service.duration_p95     25 2024-01-25 11:32:00+00:00 2024-01-25 11:56:00+00:00
722  ts-travel2-service     trace.from_ts-admin-travel-service.error_rate     25 2024-01-25 11:32:00+00:00 2024-01-25 11:56:00+00:00
723  ts-travel2-service      trace.from_ts-admin-travel-service.row_count     25 2024-01-25 11:32:00+00:00 2024-01-25 11:56:00+00:00
712   ts-travel-service          trace.to_ts-travel-service.duration_mean     25 2024-01-25 11:32:00+00:00 2024-01-25 11:56:00+00:00
713   ts-travel-service           trace.to_ts-travel-service.duration_p95     25 2024-01-25 11:32:00+00:00 2024-01-25 11:56:00+00:00
714   ts-travel-service             trace.to_ts-travel-service.error_rate     25 2024-01-25 11:32:00+00:00 2024-01-25 11:56:00+00:00
715   ts-travel-service              trace.to_ts-travel-service.row_count     25 2024-01-25 11:32:00+00:00 2024-01-25 11:56:00+00:00
716  ts-travel2-service                     trace.from_root.duration_mean     25 2024-01-25 11:32:00+00:00 2024-01-25 11:56:00+00:00
717  ts-travel2-service                      trace.from_root.duration_p95     25 2024-01-25 11:32:00+00:00 2024-01-25 11:56:00+00:00
718  ts-travel2-service                        trace.from_root.error_rate     25 2024-01-25 11:32:00+00:00 2024-01-25 11:56:00+00:00
719  ts-travel2-service                         trace.from_root.row_count     25 2024-01-25 11:32:00+00:00 2024-01-25 11:56:00+00:00,                         cmdb_id         log_name  count               earliest_ts                 latest_ts
0   ts-admin-basic-info-service  log.error_count     25 2024-01-25 11:32:00+00:00 2024-01-25 11:56:00+00:00
1   ts-admin-basic-info-service    log.row_count     25 2024-01-25 11:32:00+00:00 2024-01-25 11:56:00+00:00
2       ts-admin-travel-service  log.error_count     25 2024-01-25 11:32:00+00:00 2024-01-25 11:56:00+00:00
3       ts-admin-travel-service    log.row_count     25 2024-01-25 11:32:00+00:00 2024-01-25 11:56:00+00:00
12             ts-basic-service  log.error_count     25 2024-01-25 11:32:00+00:00 2024-01-25 11:56:00+00:00
13             ts-basic-service    log.row_count     25 2024-01-25 11:32:00+00:00 2024-01-25 11:56:00+00:00
11              ts-auth-service    log.row_count     25 2024-01-25 11:32:00+00:00 2024-01-25 11:56:00+00:00
10              ts-auth-service  log.error_count     25 2024-01-25 11:32:00+00:00 2024-01-25 11:56:00+00:00
22           ts-consign-service  log.error_count     25 2024-01-25 11:32:00+00:00 2024-01-25 11:56:00+00:00
23           ts-consign-service    log.row_count     25 2024-01-25 11:32:00+00:00 2024-01-25 11:56:00+00:00
89           ts-travel2-service    log.row_count     25 2024-01-25 11:32:00+00:00 2024-01-25 11:56:00+00:00
88           ts-travel2-service  log.error_count     25 2024-01-25 11:32:00+00:00 2024-01-25 11:56:00+00:00
85            ts-travel-service    log.row_count     25 2024-01-25 11:32:00+00:00 2024-01-25 11:56:00+00:00
84            ts-travel-service  log.error_count     25 2024-01-25 11:32:00+00:00 2024-01-25 11:56:00+00:00
62             ts-price-service  log.error_count     25 2024-01-25 11:32:00+00:00 2024-01-25 11:56:00+00:00
63             ts-price-service    log.row_count     25 2024-01-25 11:32:00+00:00 2024-01-25 11:56:00+00:00
67             ts-route-service    log.row_count     25 2024-01-25 11:32:00+00:00 2024-01-25 11:56:00+00:00
66             ts-route-service  log.error_count     25 2024-01-25 11:32:00+00:00 2024-01-25 11:56:00+00:00
68              ts-seat-service  log.error_count     25 2024-01-25 11:32:00+00:00 2024-01-25 11:56:00+00:00
69              ts-seat-service    log.row_count     25 2024-01-25 11:32:00+00:00 2024-01-25 11:56:00+00:00, Empty DataFrame
Columns: [cmdb_id, count, earliest_ts, latest_ts]
Index: [], Empty DataFrame
Columns: [ts, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[2]:

import numpy as np
import pandas as pd

# Reuse existing metric_df (assumed loaded previously) and its 'ts' datetime column.
# Filter to the five target components
targets = ["ts-auth-service", "ts-order-service", "ts-route-service", "ts-train-service", "ts-travel-service"]
metric_sel = metric_df[metric_df["cmdb_id"].isin(targets)].copy()

# 1) Compute global thresholds on the full available series for each (cmdb_id, kpi_name)
thresholds = (
    metric_sel
    .groupby(["cmdb_id", "kpi_name"], as_index=False)
    .agg(
        total_count=("value", "size"),
        global_P95=("value", lambda x: float(x.quantile(0.95)) if x.size>0 else np.nan),
        global_P5=("value", lambda x: float(x.quantile(0.05)) if x.size>0 else np.nan),
    )
)

# Incident window (UTC)
window_start = pd.to_datetime("2024-01-25 11:29:40", utc=True)
window_end = pd.to_datetime("2024-01-25 11:59:40", utc=True)

# Filter metric rows to the incident window
window_rows = metric_sel[(metric_sel["ts"] >= window_start) & (metric_sel["ts"] <= window_end)].copy()

# Merge thresholds into window rows so each window row has its group's global thresholds;
# we will perform groupwise aggregations and keep groups even if they have zero rows in window.
merged = thresholds.merge(window_rows, on=["cmdb_id", "kpi_name"], how="left", suffixes=("_th", "_row"))

# Group and compute requested within-window aggregates per group (using merged to preserve all groups)
def compute_group_stats(g):
    # g contains columns: cmdb_id, kpi_name, total_count, global_P95, global_P5, and possibly timestamp/value rows (may be NaN)
    total_count = int(g["total_count"].iat[0])
    global_P95 = float(g["global_P95"].iat[0])
    global_P5 = float(g["global_P5"].iat[0])

    vals = g["value"]
    tss = g["ts"]

    window_count = int(vals.count())  # non-NaN count within window

    # Above P95
    if window_count > 0:
        above_mask = vals > global_P95
        count_above = int(above_mask.sum())
        earliest_above = tss[above_mask].min() if count_above > 0 else pd.NaT
        max_value = float(vals.max(skipna=True)) if vals.count()>0 else np.nan
    else:
        count_above = 0
        earliest_above = pd.NaT
        max_value = np.nan

    ratio_above = np.nan if (global_P95 == 0) else ((max_value - global_P95) / global_P95) if not np.isnan(max_value) else np.nan

    # Below P5
    if window_count > 0:
        below_mask = vals < global_P5
        count_below = int(below_mask.sum())
        earliest_below = tss[below_mask].min() if count_below > 0 else pd.NaT
        min_value = float(vals.min(skipna=True)) if vals.count()>0 else np.nan
    else:
        count_below = 0
        earliest_below = pd.NaT
        min_value = np.nan

    ratio_below = np.nan if (global_P5 == 0) else ((global_P5 - min_value) / global_P5) if not np.isnan(min_value) else np.nan

    return pd.Series({
        "total_count": total_count,
        "global_P95": global_P95,
        "global_P5": global_P5,
        "window_count": window_count,
        "count_above_P95": count_above,
        "earliest_above_P95_ts": earliest_above,
        "max_value_in_window": max_value,
        "ratio_above": ratio_above,
        "count_below_P5": count_below,
        "earliest_below_P5_ts": earliest_below,
        "min_value_in_window": min_value,
        "ratio_below": ratio_below
    })

grouped_stats = merged.groupby(["cmdb_id", "kpi_name"], sort=False).apply(compute_group_stats).reset_index()

# Sort by count_above_P95 desc then count_below_P5 desc, limit to top 50 rows
result_table = grouped_stats.sort_values(["count_above_P95", "count_below_P5"], ascending=[False, False]).head(50)

# Overall summary counts
distinct_pairs = int(grouped_stats.shape[0])
breach_pairs = int(((grouped_stats["count_above_P95"] > 0) | (grouped_stats["count_below_P5"] > 0)).sum())

summary = pd.DataFrame({
    "metric_pairs_processed": [distinct_pairs],
    "pairs_with_any_breach_in_window": [breach_pairs]
})

# Display compact results: result_table (selected cols) and summary
result_table_selected = result_table[
    [
        "cmdb_id", "kpi_name",
        "total_count", "global_P95", "global_P5",
        "window_count", "count_above_P95", "earliest_above_P95_ts",
        "max_value_in_window", "ratio_above",
        "count_below_P5", "earliest_below_P5_ts",
        "min_value_in_window", "ratio_below"
    ]
].reset_index(drop=True)

result_table_selected, summary

```
Out[2]:
```
Summary (plain English)

- Processed 35 distinct (cmdb_id, kpi_name) metric pairs for the five services you asked about: ts-auth-service, ts-order-service, ts-route-service, ts-train-service, ts-travel-service.
- All 35 pairs had at least one breach in the incident window (i.e., count_above_P95>0 or count_below_P5>0).
- Most metric series had 25 total points in the full series and 25 points inside the incident window (the dataset covers the full window).
- Typical breach pattern: many metrics show a small number of outliers in the window — commonly 2 points above the global P95 and 2 points below the global P5 for a given metric.
- Example highlights (from the aggregates):
  - ts-auth-service / cpu: total_count=25, global_P95≈16.76, global_P5≈12.11; window max≈17.59 (ratio_above ≈ 4.9%); count_above_P95=2, count_below_P5=2.
  - ts-auth-service / diskio: total_count=25, global_P95≈64431, global_P5≈39760; window max≈95327 (ratio_above ≈ 47.95%); count_above_P95=2.
  - ts-travel-service / latency-90: total_count=25, global_P95≈0.5309, global_P5≈0.1088; window max≈0.7768 (ratio_above ≈ 46.3%); count_above_P95=2, count_below_P5=2.
  - Many memory (mem) metrics show only very small deviations above P95 (ratio_above ≪ 1%), but still have 1–2 points outside P5/P95.
- Overall interpretation: every monitored KPI for these five services showed short-lived anomalies in the incident window (small counts of samples exceeding global P95 or falling below global P5). Some metrics (notably certain disk I/O and latency KPIs) show relatively large excursions above their P95 values, while others (e.g., memory) have only minor deviations.

If you want, I can:
- show the top N metric pairs ranked by severity (ratio_above or ratio_below), or
- provide the full compact table (up to 50 rows) for inspection.

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id    kpi_name  total_count    global_P95     global_P5  window_count  count_above_P95     earliest_above_P95_ts  max_value_in_window  ratio_above  count_below_P5      earliest_below_P5_ts  min_value_in_window  ratio_below
0     ts-auth-service         cpu           25  1.676474e+01  1.211026e+01            25                2 2024-01-25 11:32:00+00:00         1.759280e+01     0.049393               2 2024-01-25 11:43:00+00:00         1.069699e+01     0.116700
1     ts-auth-service      diskio           25  6.443169e+04  3.976037e+04            25                2 2024-01-25 11:35:00+00:00         9.532740e+04     0.479511               2 2024-01-25 11:36:00+00:00         3.857474e+04     0.029819
2     ts-auth-service  latency-50           25  3.120495e-01  2.086881e-01            25                2 2024-01-25 11:32:00+00:00         3.571591e-01     0.144559               2 2024-01-25 11:39:00+00:00         2.076997e-01     0.004737
3     ts-auth-service  latency-90           25  8.333021e-01  4.574533e-01            25                2 2024-01-25 11:36:00+00:00         8.655709e-01     0.038724               2 2024-01-25 11:39:00+00:00         4.482230e-01     0.020178
4     ts-auth-service         mem           25  2.395296e+08  2.249990e+08            25                2 2024-01-25 11:55:00+00:00         2.396734e+08     0.000600               2 2024-01-25 11:33:00+00:00         2.227827e+08     0.009850
..                ...         ...          ...           ...           ...           ...              ...                       ...                  ...          ...             ...                       ...                  ...          ...
30  ts-travel-service  latency-90           25  5.309344e-01  1.088001e-01            25                2 2024-01-25 11:32:00+00:00         7.768277e-01     0.463133               2 2024-01-25 11:43:00+00:00         9.430905e-02     0.133190
31  ts-travel-service         mem           25  2.652889e+08  2.495662e+08            25                2 2024-01-25 11:55:00+00:00         2.655014e+08     0.000801               2 2024-01-25 11:32:00+00:00         2.443198e+08     0.021022
32  ts-travel-service      socket           25  1.756333e+01  1.385333e+01            25                2 2024-01-25 11:39:00+00:00         1.931667e+01     0.099829               2 2024-01-25 11:46:00+00:00         1.340000e+01     0.032724
33  ts-travel-service    workload           25  9.188213e+00  5.700253e+00            25                2 2024-01-25 11:38:00+00:00         9.492250e+00     0.033090               2 2024-01-25 11:32:00+00:00         4.606700e+00     0.191843
34    ts-auth-service      socket           25  1.846000e+01  1.700000e+01            25                2 2024-01-25 11:36:00+00:00         1.898333e+01     0.028350               1 2024-01-25 11:44:00+00:00         1.640000e+01     0.035294

[35 rows x 14 columns],    metric_pairs_processed  pairs_with_any_breach_in_window
0                      35                               35)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import numpy as np
import pandas as pd

# Reuse metric_df and its 'ts' column from prior execution
targets = ["ts-auth-service", "ts-order-service", "ts-route-service", "ts-train-service", "ts-travel-service"]
metric_sel = metric_df[metric_df["cmdb_id"].isin(targets)].copy()

# 1) Compute global P95 and P5 per (cmdb_id, kpi_name) using full series
thresholds = (
    metric_sel
    .groupby(["cmdb_id", "kpi_name"], as_index=False)
    .agg(
        total_count=("value", "size"),
        global_P95=("value", lambda x: float(x.quantile(0.95)) if x.size > 0 else np.nan),
        global_P5=("value", lambda x: float(x.quantile(0.05)) if x.size > 0 else np.nan),
    )
)

# 2) Filter metric rows to the incident window (UTC)
window_start = pd.to_datetime("2024-01-25 11:29:40", utc=True)
window_end = pd.to_datetime("2024-01-25 11:59:40", utc=True)
window_rows = metric_sel[(metric_sel["ts"] >= window_start) & (metric_sel["ts"] <= window_end)].copy()

# Helper to compute longest consecutive runs and earliest start timestamp
def longest_run_info(bool_series, ts_series):
    # bool_series and ts_series are aligned pandas Series (ordered by ts)
    longest_len = 0
    longest_start = pd.NaT
    cur_len = 0
    cur_start = None
    for flag, ts in zip(bool_series.fillna(False).tolist(), ts_series.tolist()):
        if flag:
            if cur_len == 0:
                cur_start = ts
            cur_len += 1
        else:
            if cur_len > longest_len:
                longest_len = cur_len
                longest_start = cur_start
            cur_len = 0
            cur_start = None
    # finalize at end
    if cur_len > longest_len:
        longest_len = cur_len
        longest_start = cur_start
    return int(longest_len), (longest_start if not pd.isna(longest_start) else pd.NaT)

# 3) For each group compute requested window metrics
rows = []
for _, row in thresholds.iterrows():
    cmdb = row["cmdb_id"]
    kpi = row["kpi_name"]
    g_p95 = row["global_P95"]
    g_p5 = row["global_P5"]
    total_count = int(row["total_count"])

    grp = window_rows[(window_rows["cmdb_id"] == cmdb) & (window_rows["kpi_name"] == kpi)].copy()
    grp = grp.sort_values("ts")
    window_count = int(grp.shape[0])

    if window_count == 0:
        # No data in window
        longest_above = 0
        earliest_above = pd.NaT
        max_val = np.nan
        ratio_above = np.nan

        longest_below = 0
        earliest_below = pd.NaT
        min_val = np.nan
        ratio_below = np.nan
    else:
        vals = grp["value"]
        tss = grp["ts"]

        # Above P95
        above_mask = vals > g_p95
        longest_above, earliest_above = longest_run_info(above_mask, tss)
        max_val = float(vals.max(skipna=True)) if vals.count()>0 else np.nan
        ratio_above = np.nan if (g_p95 == 0) else ((max_val - g_p95) / g_p95) if not np.isnan(max_val) else np.nan

        # Below P5
        below_mask = vals < g_p5
        longest_below, earliest_below = longest_run_info(below_mask, tss)
        min_val = float(vals.min(skipna=True)) if vals.count()>0 else np.nan
        ratio_below = np.nan if (g_p5 == 0) else ((g_p5 - min_val) / g_p5) if not np.isnan(min_val) else np.nan

    rows.append({
        "cmdb_id": cmdb,
        "kpi_name": kpi,
        "total_count": total_count,
        "global_P95": g_p95,
        "global_P5": g_p5,
        "window_count": window_count,
        "longest_consecutive_above": longest_above,
        "earliest_start_above_ts": earliest_above,
        "max_value_in_window": max_val,
        "ratio_above": ratio_above,
        "longest_consecutive_below": longest_below,
        "earliest_start_below_ts": earliest_below,
        "min_value_in_window": min_val,
        "ratio_below": ratio_below
    })

agg_df = pd.DataFrame(rows)

# 4) Sort by max(longest_consecutive_above, longest_consecutive_below) desc then by max(ratio_above, ratio_below) desc
agg_df["max_longest"] = agg_df[["longest_consecutive_above", "longest_consecutive_below"]].max(axis=1)
agg_df["max_ratio"] = agg_df[["ratio_above", "ratio_below"]].replace([np.inf, -np.inf], np.nan).abs().max(axis=1)

result_table = (
    agg_df
    .sort_values(["max_longest", "max_ratio"], ascending=[False, False])
    .drop(columns=["max_longest", "max_ratio"])
    .reset_index(drop=True)
    .head(20)
)

# 5) Summary counts for runs >=3
count_above_ge3 = int((agg_df["longest_consecutive_above"] >= 3).sum())
count_below_ge3 = int((agg_df["longest_consecutive_below"] >= 3).sum())
summary_counts = pd.DataFrame({
    "pairs_with_longest_consecutive_above_>=3": [count_above_ge3],
    "pairs_with_longest_consecutive_below_>=3": [count_below_ge3],
    "total_pairs_processed": [int(agg_df.shape[0])]
})

# Select compact columns for display
result_table_selected = result_table[
    [
        "cmdb_id", "kpi_name",
        "total_count", "global_P95", "global_P5",
        "window_count",
        "longest_consecutive_above", "earliest_start_above_ts", "max_value_in_window", "ratio_above",
        "longest_consecutive_below", "earliest_start_below_ts", "min_value_in_window", "ratio_below"
    ]
]

result_table_selected, summary_counts

```
Out[3]:
```
Summary (plain English)

- Data processed: 35 distinct (cmdb_id, kpi_name) metric pairs for the five services.
- Window used: 2024-01-25 11:29:40 — 2024-01-25 11:59:40 UTC. Most groups had 25 points in the window.
- Longest consecutive breach runs:
  - No metric had a prolonged run of >= 3 consecutive samples above its global P95 or below its global P5. Counts:
    - pairs with longest_consecutive_above >= 3: 0
    - pairs with longest_consecutive_below >= 3: 0
- Short run / spike behavior (top examples from the ordered table):
  - ts-travel-service / latency-50: longest_above = 2 (start 2024-01-25 11:32:00), max_in_window ≈ 0.1157, ratio_above ≈ 3.12 (very large relative excursion because global_P95 is small).
  - ts-train-service / latency-90: longest_above = 1, longest_below = 2 (earliest below start 11:52), max_in_window ≈ 0.0596, ratio_above ≈ 1.34.
  - ts-order-service / cpu: longest_above = 2 (start 11:32), max_in_window ≈ 12.79, ratio_above ≈ 0.85.
  - ts-train-service / latency-50: longest_above = 2, ratio_above ≈ 0.73.
  - ts-order-service / latency-50: longest_above = 2, longest_below = 2, ratio_above ≈ 0.53.
  - Several diskio and latency KPIs show noticeable single/multi-sample excursions above P95 (ratios vary from small fractions to >3 for low-valued latency metrics).
- Overall interpretation: many KPIs experienced brief spikes or short dips within the incident window (typically runs of length 1–2), but there were no sustained runs (>=3 consecutive samples) above P95 or below P5 for any of the monitored KPI pairs. Some spikes are large in relative terms (especially for small-valued latency metrics), while others (e.g., memory metrics) show only minor deviations.

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id    kpi_name  total_count    global_P95     global_P5  window_count  longest_consecutive_above   earliest_start_above_ts  max_value_in_window  ratio_above  longest_consecutive_below   earliest_start_below_ts  min_value_in_window  ratio_below
0   ts-travel-service  latency-50           25  2.806696e-02  1.907951e-02            25                          2 2024-01-25 11:32:00+00:00         1.156743e-01     3.121370                          1 2024-01-25 11:43:00+00:00         1.875385e-02     0.017069
1    ts-train-service  latency-90           25  2.548797e-02  8.998440e-03            25                          1 2024-01-25 11:32:00+00:00         5.959172e-02     1.338033                          2 2024-01-25 11:52:00+00:00         8.958455e-03     0.004444
2    ts-order-service         cpu           25  6.923253e+00  1.530153e+00            25                          2 2024-01-25 11:32:00+00:00         1.279280e+01     0.847801                          1 2024-01-25 11:52:00+00:00         1.062388e+00     0.305698
3    ts-train-service  latency-50           25  8.150083e-03  4.644937e-03            25                          2 2024-01-25 11:32:00+00:00         1.411401e-02     0.731763                          2 2024-01-25 11:53:00+00:00         4.538491e-03     0.022917
4    ts-order-service  latency-50           25  1.597457e-02  7.447384e-03            25                          2 2024-01-25 11:32:00+00:00         2.443439e-02     0.529581                          2 2024-01-25 11:52:00+00:00         7.446008e-03     0.000185
5   ts-travel-service  latency-90           25  5.309344e-01  1.088001e-01            25                          2 2024-01-25 11:32:00+00:00         7.768277e-01     0.463133                          1 2024-01-25 11:43:00+00:00         9.430905e-02     0.133190
6   ts-travel-service         cpu           25  1.332223e+01  3.076801e+00            25                          2 2024-01-25 11:32:00+00:00         1.855160e+01     0.392530                          1 2024-01-25 11:48:00+00:00         2.928121e+00     0.048323
7    ts-route-service  latency-50           25  7.727092e-03  3.714275e-03            25                          2 2024-01-25 11:32:00+00:00         9.785965e-03     0.266449                          1 2024-01-25 11:42:00+00:00         3.504077e-03     0.056592
8   ts-travel-service      diskio           25  1.491222e+05  4.452509e+04            25                          2 2024-01-25 11:50:00+00:00         1.809871e+05     0.213683                          1 2024-01-25 11:38:00+00:00         4.278263e+04     0.039134
9    ts-train-service    workload           25  1.953117e+01  1.664289e+01            25                          2 2024-01-25 11:51:00+00:00         2.040237e+01     0.044606                          1 2024-01-25 11:32:00+00:00         1.337315e+01     0.196465
10    ts-auth-service  latency-50           25  3.120495e-01  2.086881e-01            25                          2 2024-01-25 11:32:00+00:00         3.571591e-01     0.144559                          1 2024-01-25 11:39:00+00:00         2.076997e-01     0.004737
11    ts-auth-service         cpu           25  1.676474e+01  1.211026e+01            25                          2 2024-01-25 11:32:00+00:00         1.759280e+01     0.049393                          1 2024-01-25 11:43:00+00:00         1.069699e+01     0.116700
12   ts-route-service      diskio           25  1.725946e+10  4.620423e+04            25                          2 2024-01-25 11:53:00+00:00         1.764342e+10     0.022247                          1 2024-01-25 11:35:00+00:00         4.134966e+04     0.105068
13  ts-travel-service      socket           25  1.756333e+01  1.385333e+01            25                          1 2024-01-25 11:39:00+00:00         1.931667e+01     0.099829                          2 2024-01-25 11:46:00+00:00         1.340000e+01     0.032724
14   ts-route-service    workload           25  3.160667e+01  2.732224e+01            25                          2 2024-01-25 11:51:00+00:00         3.185893e+01     0.007981                          1 2024-01-25 11:32:00+00:00         2.485010e+01     0.090481
15   ts-train-service         mem           25  2.499473e+08  2.444181e+08            25                          1 2024-01-25 11:32:00+00:00         2.648857e+08     0.059766                          2 2024-01-25 11:33:00+00:00         2.423442e+08     0.008485
16    ts-auth-service  latency-90           25  8.333021e-01  4.574533e-01            25                          2 2024-01-25 11:36:00+00:00         8.655709e-01     0.038724                          1 2024-01-25 11:39:00+00:00         4.482230e-01     0.020178
17   ts-route-service         cpu           25  3.371631e+01  5.392680e+00            25                          1 2024-01-25 11:47:00+00:00         3.473058e+01     0.030083                          2 2024-01-25 11:43:00+00:00         5.296151e+00     0.017900
18   ts-route-service      socket           25  1.591000e+01  1.188333e+01            25                          2 2024-01-25 11:45:00+00:00         1.626667e+01     0.022418                          2 2024-01-25 11:42:00+00:00         1.171667e+01     0.014025
19  ts-travel-service         mem           25  2.652889e+08  2.495662e+08            25                          2 2024-01-25 11:55:00+00:00         2.655014e+08     0.000801                          2 2024-01-25 11:32:00+00:00         2.443198e+08     0.021022,    pairs_with_longest_consecutive_above_>=3  pairs_with_longest_consecutive_below_>=3  total_pairs_processed
0                                         0                                         0                     35)```
```

In [ ]:
# In[4]:

import numpy as np
import pandas as pd

# Reuse trace_df and its 'ts' datetime column from prior execution.
targets = ["ts-auth-service", "ts-order-service", "ts-route-service", "ts-train-service", "ts-travel-service"]
trace_sel = trace_df[trace_df["cmdb_id"].isin(targets)].copy()

# 1) Compute global P95 and P5 per (cmdb_id, trace_name) using full series (no window filtering)
trace_thresholds = (
    trace_sel
    .groupby(["cmdb_id", "trace_name"], as_index=False)
    .agg(
        total_count=("value", "size"),
        global_P95=("value", lambda x: float(x.quantile(0.95)) if x.size > 0 else np.nan),
        global_P5=("value", lambda x: float(x.quantile(0.05)) if x.size > 0 else np.nan),
    )
)

# 2) Filter trace rows to the incident window (UTC)
# Reuse window_start/window_end if present, otherwise define
try:
    window_start, window_end
except NameError:
    window_start = pd.to_datetime("2024-01-25 11:29:40", utc=True)
    window_end = pd.to_datetime("2024-01-25 11:59:40", utc=True)

trace_window = trace_sel[(trace_sel["ts"] >= window_start) & (trace_sel["ts"] <= window_end)].copy()

# Helper: compute longest consecutive run and earliest start timestamp
def longest_run_info(bool_series, ts_series):
    longest_len = 0
    longest_start = pd.NaT
    cur_len = 0
    cur_start = pd.NaT
    # Ensure alignment and fill NA as False for mask
    for flag, ts in zip(bool_series.fillna(False).tolist(), ts_series.tolist()):
        if flag:
            if cur_len == 0:
                cur_start = ts
            cur_len += 1
        else:
            if cur_len > longest_len:
                longest_len = cur_len
                longest_start = cur_start
            cur_len = 0
            cur_start = pd.NaT
    # finalize
    if cur_len > longest_len:
        longest_len = cur_len
        longest_start = cur_start
    return int(longest_len), (longest_start if not pd.isna(longest_start) else pd.NaT)

# 3) For each (cmdb_id, trace_name) compute requested windowed aggregates
rows = []
for _, r in trace_thresholds.iterrows():
    cmdb = r["cmdb_id"]
    tname = r["trace_name"]
    g_p95 = r["global_P95"]
    g_p5 = r["global_P5"]
    total_count = int(r["total_count"])

    grp = trace_window[(trace_window["cmdb_id"] == cmdb) & (trace_window["trace_name"] == tname)].copy()
    grp = grp.sort_values("ts")
    window_count = int(grp.shape[0])

    if window_count == 0:
        count_above = 0
        earliest_above = pd.NaT
        longest_above = 0
        max_val = np.nan
        ratio_above = np.nan

        count_below = 0
        earliest_below = pd.NaT
        longest_below = 0
        min_val = np.nan
        ratio_below = np.nan
    else:
        vals = grp["value"]
        tss = grp["ts"]

        above_mask = vals > g_p95
        count_above = int(above_mask.sum())
        earliest_above = tss[above_mask].min() if count_above > 0 else pd.NaT
        longest_above, earliest_longest_above = longest_run_info(above_mask, tss)

        below_mask = vals < g_p5
        count_below = int(below_mask.sum())
        earliest_below = tss[below_mask].min() if count_below > 0 else pd.NaT
        longest_below, earliest_longest_below = longest_run_info(below_mask, tss)

        max_val = float(vals.max(skipna=True)) if vals.count() > 0 else np.nan
        min_val = float(vals.min(skipna=True)) if vals.count() > 0 else np.nan

        ratio_above = np.nan if (g_p95 == 0) else ((max_val - g_p95) / g_p95) if not np.isnan(max_val) else np.nan
        ratio_below = np.nan if (g_p5 == 0) else ((g_p5 - min_val) / g_p5) if not np.isnan(min_val) else np.nan

        # For earliest_start_above/below prefer the start of the longest run; if longest==0 fallback to earliest single above/below
        earliest_above = earliest_longest_above if longest_above > 0 else earliest_above
        earliest_below = earliest_longest_below if longest_below > 0 else earliest_below

    rows.append({
        "cmdb_id": cmdb,
        "trace_name": tname,
        "total_count": total_count,
        "global_P95": g_p95,
        "global_P5": g_p5,
        "window_count": window_count,
        "count_above_P95": count_above,
        "earliest_above_P95_ts": earliest_above,
        "longest_consecutive_above": longest_above,
        "count_below_P5": count_below,
        "earliest_below_P5_ts": earliest_below,
        "longest_consecutive_below": longest_below,
        "max_value_in_window": max_val,
        "min_value_in_window": min_val,
        "ratio_above": ratio_above,
        "ratio_below": ratio_below
    })

trace_agg = pd.DataFrame(rows)

# 4) Sort by max(longest_consecutive_above, longest_consecutive_below) desc then by max(|ratio_above|,|ratio_below|) desc
trace_agg["max_longest"] = trace_agg[["longest_consecutive_above", "longest_consecutive_below"]].max(axis=1)
trace_agg["max_ratio_abs"] = trace_agg[["ratio_above", "ratio_below"]].abs().max(axis=1)

trace_result = (
    trace_agg
    .sort_values(["max_longest", "max_ratio_abs"], ascending=[False, False])
    .drop(columns=["max_longest", "max_ratio_abs"])
    .reset_index(drop=True)
    .head(20)
)

# 5) Counts for runs >=3
count_above_ge3 = int((trace_agg["longest_consecutive_above"] >= 3).sum())
count_below_ge3 = int((trace_agg["longest_consecutive_below"] >= 3).sum())
trace_summary = pd.DataFrame({
    "pairs_with_longest_consecutive_above_>=3": [count_above_ge3],
    "pairs_with_longest_consecutive_below_>=3": [count_below_ge3],
    "total_pairs_processed": [int(trace_agg.shape[0])]
})

# Select compact columns for display
trace_result_selected = trace_result[
    [
        "cmdb_id", "trace_name", "total_count", "global_P95", "global_P5",
        "window_count", "count_above_P95", "earliest_above_P95_ts", "longest_consecutive_above",
        "count_below_P5", "earliest_below_P5_ts", "longest_consecutive_below",
        "max_value_in_window", "min_value_in_window", "ratio_above", "ratio_below"
    ]
]

trace_result_selected, trace_summary

```
Out[4]:
```
Summary (plain English)

- Data processed: 136 distinct (cmdb_id, trace_name) trace pairs for the five services.
- Window used: 2024-01-25 11:29:40 — 2024-01-25 11:59:40 UTC; most groups have ~25 samples in the window.
- Sustained runs: none of the 136 trace pairs had a sustained run of >= 3 consecutive samples above global P95 or below global P5.
  - pairs with longest_consecutive_above >= 3: 0
  - pairs with longest_consecutive_below >= 3: 0
- Anomaly pattern: many trace features show short spikes or short dips (typically runs of length 1–2) within the window. Relative excursions can be large for small-valued duration metrics (ratio_above sometimes >> 0).
- Top examples (from the ordered aggregates):
  - ts-travel-service / trace.from_root.duration_mean: total_count=25, window_count=25, count_above_P95=2, longest_consecutive_above=2 (earliest above 2024-01-25 11:32:00), max_in_window≈2.0631, ratio_above≈0.826.
  - ts-travel-service / trace.from_ts-admin-travel-service.duration_mean: similar pattern, ratio_above≈0.819.
  - ts-order-service / trace.from_ts-seat-service.duration_mean: longest_above=2, ratio_above≈0.778.
  - Several trace row_count and duration metrics across ts-order, ts-route, ts-train and ts-travel show 1–2 sample breaches above P95 and/or below P5.
- Interpretation: traces show brief, short-lived anomalies (spikes or dips) rather than sustained degradations in the incident window. If you want, I can return the top 20 pairs table sorted by run length / ratio for closer inspection.

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id                                        trace_name  total_count  global_P95  global_P5  window_count  count_above_P95     earliest_above_P95_ts  longest_consecutive_above  count_below_P5      earliest_below_P5_ts  longest_consecutive_below  max_value_in_window  min_value_in_window  ratio_above  ratio_below
0   ts-travel-service                     trace.from_root.duration_mean           25    1.129656   0.572293            25                2 2024-01-25 11:32:00+00:00                          2               2 2024-01-25 11:40:00+00:00                          1             2.063073             0.495018     0.826283     0.135027
1   ts-travel-service  trace.from_ts-admin-travel-service.duration_mean           25    0.177502   0.067120            25                2 2024-01-25 11:32:00+00:00                          2               2 2024-01-25 11:46:00+00:00                          1             0.322877             0.064945     0.819000     0.032400
2    ts-order-service          trace.from_ts-seat-service.duration_mean           25    0.011060   0.004877            25                2 2024-01-25 11:32:00+00:00                          2               2 2024-01-25 11:48:00+00:00                          1             0.019665             0.004734     0.778012     0.029187
3    ts-route-service             trace.from_ts-basic-service.row_count           25  121.800000  80.800000            25                2 2024-01-25 11:52:00+00:00                          2               2 2024-01-25 11:32:00+00:00                          1           127.000000            32.000000     0.042693     0.603960
4    ts-train-service             trace.from_ts-basic-service.row_count           25  121.800000  80.800000            25                2 2024-01-25 11:52:00+00:00                          2               2 2024-01-25 11:32:00+00:00                          1           127.000000            32.000000     0.042693     0.603960
5   ts-travel-service          trace.from_ts-seat-service.duration_mean           25    0.031809   0.012629            25                2 2024-01-25 11:32:00+00:00                          2               2 2024-01-25 11:50:00+00:00                          2             0.045600             0.011695     0.433552     0.073953
6   ts-travel-service      trace.from_ts-preserve-service.duration_mean           22    0.561312   0.257797            22                2 2024-01-25 11:33:00+00:00                          1               2 2024-01-25 11:50:00+00:00                          2             0.743818             0.254471     0.325140     0.012902
7   ts-travel-service         trace.from_ts-travel-service.duration_p95           25    0.106786   0.051058            25                2 2024-01-25 11:32:00+00:00                          2               2 2024-01-25 11:51:00+00:00                          1             0.137503             0.048341     0.287659     0.053214
8   ts-travel-service           trace.to_ts-travel-service.duration_p95           25    0.106786   0.051058            25                2 2024-01-25 11:32:00+00:00                          2               2 2024-01-25 11:51:00+00:00                          1             0.137503             0.048341     0.287659     0.053214
9   ts-travel-service        trace.from_ts-travel-service.duration_mean           25    0.022886   0.011528            25                2 2024-01-25 11:32:00+00:00                          2               2 2024-01-25 11:40:00+00:00                          1             0.028952             0.010843     0.265061     0.059429
10  ts-travel-service          trace.to_ts-travel-service.duration_mean           25    0.022886   0.011528            25                2 2024-01-25 11:32:00+00:00                          2               2 2024-01-25 11:40:00+00:00                          1             0.028952             0.010843     0.265061     0.059429
11   ts-order-service          trace.to_ts-station-service.duration_p95           25    0.131620   0.028240            25                2 2024-01-25 11:54:00+00:00                          1               2 2024-01-25 11:32:00+00:00                          2             0.138217             0.021110     0.050123     0.252485
12   ts-order-service        trace.from_ts-travel-service.duration_mean           25    0.009567   0.005079            25                2 2024-01-25 11:32:00+00:00                          2               2 2024-01-25 11:48:00+00:00                          1             0.011886             0.005021     0.242378     0.011453
13  ts-travel-service           trace.to_ts-order-service.duration_mean           25    0.009567   0.005079            25                2 2024-01-25 11:32:00+00:00                          2               2 2024-01-25 11:48:00+00:00                          1             0.011886             0.005021     0.242378     0.011453
14   ts-order-service                     trace.from_root.duration_mean           25    0.126333   0.048132            25                2 2024-01-25 11:54:00+00:00                          1               2 2024-01-25 11:32:00+00:00                          2             0.142043             0.037517     0.124352     0.220539
15   ts-order-service       trace.from_ts-preserve-service.duration_p95           22    0.060084   0.007661            22                2 2024-01-25 11:38:00+00:00                          2               2 2024-01-25 11:53:00+00:00                          1             0.071296             0.007477     0.186614     0.024074
16  ts-travel-service            trace.to_ts-seat-service.duration_mean           25    0.125242   0.048367            25                2 2024-01-25 11:32:00+00:00                          2               2 2024-01-25 11:48:00+00:00                          1             0.148166             0.046731     0.183040     0.033837
17   ts-order-service       trace.from_ts-security-service.duration_p95           24    0.013258   0.005681            24                2 2024-01-25 11:37:00+00:00                          2               2 2024-01-25 11:48:00+00:00                          1             0.015595             0.005588     0.176349     0.016288
18   ts-order-service         trace.to_ts-station-service.duration_mean           25    0.114662   0.025442            25                2 2024-01-25 11:54:00+00:00                          1               2 2024-01-25 11:32:00+00:00                          2             0.129980             0.021110     0.133599     0.170283
19    ts-auth-service                      trace.from_root.duration_p95           25    0.916369   0.475809            25                2 2024-01-25 11:35:00+00:00                          2               2 2024-01-25 11:39:00+00:00                          1             1.068305             0.402750     0.165803     0.153548,    pairs_with_longest_consecutive_above_>=3  pairs_with_longest_consecutive_below_>=3  total_pairs_processed
0                                         0                                         0                    136)```
```

In [ ]:
# In[5]:

import numpy as np
import pandas as pd

# Reuse existing log_df and window_start/window_end from prior state
targets = ["ts-auth-service", "ts-order-service", "ts-route-service", "ts-train-service", "ts-travel-service"]
log_sel = log_df[log_df["cmdb_id"].isin(targets)].copy()

# 1) Compute global thresholds on full series per (cmdb_id, log_name)
log_thresholds = (
    log_sel
    .groupby(["cmdb_id", "log_name"], as_index=False)
    .agg(
        total_count=("value", "size"),
        global_P95=("value", lambda x: float(x.quantile(0.95)) if x.size>0 else np.nan),
        global_P90=("value", lambda x: float(x.quantile(0.90)) if x.size>0 else np.nan),
        global_P5 =("value", lambda x: float(x.quantile(0.05)) if x.size>0 else np.nan),
    )
)

# 2) Filter log rows to the incident window (window_start/window_end should exist)
try:
    window_start, window_end
except NameError:
    window_start = pd.to_datetime("2024-01-25 11:29:40", utc=True)
    window_end = pd.to_datetime("2024-01-25 11:59:40", utc=True)

log_window = log_sel[(log_sel["ts"] >= window_start) & (log_sel["ts"] <= window_end)].copy()
log_window = log_window.sort_values(["cmdb_id", "log_name", "ts"])

# Helper to compute longest consecutive run and earliest start timestamp for a boolean mask series aligned with ts
def longest_run_info(bool_series, ts_series):
    longest_len = 0
    longest_start = pd.NaT
    cur_len = 0
    cur_start = pd.NaT
    for flag, ts in zip(bool_series.fillna(False).tolist(), ts_series.tolist()):
        if flag:
            if cur_len == 0:
                cur_start = ts
            cur_len += 1
        else:
            if cur_len > longest_len:
                longest_len = cur_len
                longest_start = cur_start
            cur_len = 0
            cur_start = pd.NaT
    if cur_len > longest_len:
        longest_len = cur_len
        longest_start = cur_start
    return int(longest_len), (longest_start if not pd.isna(longest_start) else pd.NaT)

# 3) Compute windowed aggregates per pair
rows = []
for _, th in log_thresholds.iterrows():
    cmdb = th["cmdb_id"]
    lname = th["log_name"]
    g_p95 = th["global_P95"]
    g_p90 = th["global_P90"]
    g_p5 = th["global_P5"]
    total_count = int(th["total_count"])

    grp = log_window[(log_window["cmdb_id"] == cmdb) & (log_window["log_name"] == lname)].copy()
    grp = grp.sort_values("ts")
    window_count = int(grp.shape[0])

    if window_count == 0:
        count_above_p95 = 0
        earliest_above_p95 = pd.NaT
        longest_above_p95 = 0

        count_above_p90 = 0
        longest_above_p90 = 0

        count_below_p5 = 0
        earliest_below_p5 = pd.NaT
        longest_below_p5 = 0

        max_val = np.nan
        min_val = np.nan
        ratio_above = np.nan
        ratio_below = np.nan
    else:
        vals = grp["value"]
        tss = grp["ts"]

        # Above P95
        above_p95_mask = vals > g_p95
        count_above_p95 = int(above_p95_mask.sum())
        earliest_above_p95 = tss[above_p95_mask].min() if count_above_p95>0 else pd.NaT
        longest_above_p95, earliest_longest_above_p95 = longest_run_info(above_p95_mask, tss)
        if longest_above_p95>0:
            earliest_above_p95 = earliest_longest_above_p95

        # Above P90
        above_p90_mask = vals > g_p90
        count_above_p90 = int(above_p90_mask.sum())
        longest_above_p90, _ = longest_run_info(above_p90_mask, tss)

        # Below P5
        below_p5_mask = vals < g_p5
        count_below_p5 = int(below_p5_mask.sum())
        earliest_below_p5 = tss[below_p5_mask].min() if count_below_p5>0 else pd.NaT
        longest_below_p5, earliest_longest_below_p5 = longest_run_info(below_p5_mask, tss)
        if longest_below_p5>0:
            earliest_below_p5 = earliest_longest_below_p5

        max_val = float(vals.max(skipna=True)) if vals.count()>0 else np.nan
        min_val = float(vals.min(skipna=True)) if vals.count()>0 else np.nan
        ratio_above = np.nan if (g_p95==0) else ((max_val - g_p95)/g_p95) if not np.isnan(max_val) else np.nan
        ratio_below = np.nan if (g_p5==0) else ((g_p5 - min_val)/g_p5) if not np.isnan(min_val) else np.nan

    rows.append({
        "cmdb_id": cmdb,
        "log_name": lname,
        "total_count": total_count,
        "global_P95": g_p95,
        "global_P90": g_p90,
        "global_P5": g_p5,
        "window_count": window_count,
        "count_above_P95": count_above_p95,
        "earliest_above_P95_ts": earliest_above_p95,
        "longest_consecutive_above_P95": longest_above_p95,
        "count_above_P90": count_above_p90,
        "longest_consecutive_above_P90": longest_above_p90,
        "count_below_P5": count_below_p5,
        "earliest_below_P5_ts": earliest_below_p5,
        "longest_consecutive_below_P5": longest_below_p5,
        "max_value_in_window": max_val,
        "min_value_in_window": min_val,
        "ratio_above": ratio_above,
        "ratio_below": ratio_below
    })

log_agg = pd.DataFrame(rows)

# 4) Sort by longest_consecutive_above_P90 desc then by max(ratio_above, ratio_below) desc
log_agg["max_ratio"] = log_agg[["ratio_above", "ratio_below"]].replace([np.inf, -np.inf], np.nan).max(axis=1)
result_table = (
    log_agg
    .sort_values(["longest_consecutive_above_P90", "max_ratio"], ascending=[False, False])
    .reset_index(drop=True)
    .head(20)
)

# 5) Summary counts
count_above_p95_ge3 = int((log_agg["longest_consecutive_above_P95"] >= 3).sum())
count_above_p90_ge3 = int((log_agg["longest_consecutive_above_P90"] >= 3).sum())
count_below_p5_ge3 = int((log_agg["longest_consecutive_below_P5"] >= 3).sum())
summary_counts = pd.DataFrame({
    "pairs_with_longest_consecutive_above_P95_>=3": [count_above_p95_ge3],
    "pairs_with_longest_consecutive_above_P90_>=3": [count_above_p90_ge3],
    "pairs_with_longest_consecutive_below_P5_>=3": [count_below_p5_ge3],
    "total_pairs_processed": [int(log_agg.shape[0])]
})

# Select compact columns for display
result_table_selected = result_table[
    [
        "cmdb_id", "log_name", "total_count", "global_P95", "global_P90", "global_P5",
        "window_count",
        "count_above_P95", "earliest_above_P95_ts", "longest_consecutive_above_P95",
        "count_above_P90", "longest_consecutive_above_P90",
        "count_below_P5", "earliest_below_P5_ts", "longest_consecutive_below_P5",
        "max_value_in_window", "min_value_in_window", "ratio_above", "ratio_below"
    ]
]

result_table_selected, summary_counts

```
Out[5]:
```
Summary (plain English)

- Data processed: 8 distinct (cmdb_id, log_name) pairs for the five services, evaluated over the window 2024-01-25 11:29:40 — 2024-01-25 11:59:40 UTC.
- Global thresholds were computed on the full series (P95, P90, P5) before window filtering.

Key findings (compact):
- log.row_count for several services showed short spikes/dips:
  - ts-route-service / log.row_count: total_count=25, window_count=25, count_above_P95=2 (earliest above 11:44), longest_consecutive_above_P95=1, max_in_window=939, ratio_above ≈ 0.020; count_below_P5=2 (earliest below 11:32), min_in_window=258, ratio_below ≈ 0.588.
  - ts-auth-service / log.row_count: count_above_P95=2 (earliest 11:40), count_above_P90=3, longest_consecutive_above_P95=1, max=320, ratio_above ≈ 0.040; count_below_P5=2 (earliest 11:32), min=88, ratio_below ≈ 0.575.
  - ts-order-service / log.row_count: count_above_P95=2 (earliest 11:51), max=386, ratio_above ≈ 0.075; count_below_P5=2 (earliest 11:32), min=84, ratio_below ≈ 0.550.
  - ts-travel-service / log.row_count: count_above_P95=2 (earliest 11:51), count_above_P90=3, max=1782, ratio_above ≈ 0.079; count_below_P5=2 (earliest 11:32), min=431, ratio_below ≈ 0.547.
- log.error_count for all these services is zero across the series (global thresholds are 0 and window values are 0) — no error log activity to inspect.

Sustained-run summary:
- Number of pairs with longest_consecutive_above_P95 >= 3: 0
- Number of pairs with longest_consecutive_above_P90 >= 3: 0
- Number of pairs with longest_consecutive_below_P5 >= 3: 0

Interpretation:
- There are short, brief deviations in log.row_count (typically 1–2 consecutive samples above P95 or below P5), but no sustained (>=3) consecutive breaches. Error counts show no activity. If you want, I can return the full compact table (top 20) for closer inspection.

The original code execution output of IPython Kernel is also provided below for reference:

(             cmdb_id         log_name  total_count  global_P95  global_P90  global_P5  window_count  count_above_P95     earliest_above_P95_ts  longest_consecutive_above_P95  count_above_P90  longest_consecutive_above_P90  count_below_P5      earliest_below_P5_ts  longest_consecutive_below_P5  max_value_in_window  min_value_in_window  ratio_above  ratio_below
0   ts-route-service    log.row_count           25       920.4       910.0      626.8            25                2 2024-01-25 11:44:00+00:00                              1                2                              1               2 2024-01-25 11:32:00+00:00                             1                939.0                258.0     0.020209     0.588385
1    ts-auth-service    log.row_count           25       307.6       305.2      207.2            25                2 2024-01-25 11:40:00+00:00                              1                3                              1               2 2024-01-25 11:32:00+00:00                             1                320.0                 88.0     0.040312     0.575290
2   ts-order-service    log.row_count           25       359.2       348.0      186.8            25                2 2024-01-25 11:51:00+00:00                              1                2                              1               2 2024-01-25 11:32:00+00:00                             1                386.0                 84.0     0.074610     0.550321
3  ts-travel-service    log.row_count           25      1650.8      1619.6      950.4            25                2 2024-01-25 11:51:00+00:00                              1                3                              1               2 2024-01-25 11:32:00+00:00                             1               1782.0                431.0     0.079477     0.546507
4    ts-auth-service  log.error_count           25         0.0         0.0        0.0            25                0                       NaT                              0                0                              0               0                       NaT                             0                  0.0                  0.0          NaN          NaN
5   ts-order-service  log.error_count           25         0.0         0.0        0.0            25                0                       NaT                              0                0                              0               0                       NaT                             0                  0.0                  0.0          NaN          NaN
6   ts-route-service  log.error_count           25         0.0         0.0        0.0            25                0                       NaT                              0                0                              0               0                       NaT                             0                  0.0                  0.0          NaN          NaN
7  ts-travel-service  log.error_count           25         0.0         0.0        0.0            25                0                       NaT                              0                0                              0               0                       NaT                             0                  0.0                  0.0          NaN          NaN,    pairs_with_longest_consecutive_above_P95_>=3  pairs_with_longest_consecutive_above_P90_>=3  pairs_with_longest_consecutive_below_P5_>=3  total_pairs_processed
0                                             0                                             0                                            0                      8)```
```

In [ ]:
# In[6]:

import numpy as np
import pandas as pd

# Reuse variables: metric_df, trace_df, log_df, window_start, window_end
targets = ["ts-auth-service", "ts-order-service", "ts-route-service", "ts-train-service", "ts-travel-service"]

# Ensure window_start/window_end exist
try:
    window_start, window_end
except NameError:
    window_start = pd.to_datetime("2024-01-25 11:29:40", utc=True)
    window_end = pd.to_datetime("2024-01-25 11:59:40", utc=True)

def process_source(df, name_col):
    # Restrict to target services
    sel = df[df["cmdb_id"].isin(targets)].copy()
    # Compute global thresholds on full series per (cmdb_id, name_col)
    thresholds = (
        sel
        .groupby(["cmdb_id", name_col], as_index=False)
        .agg(
            total_count=("value", "size"),
            global_P95=("value", lambda x: float(x.quantile(0.95)) if x.size>0 else np.nan),
            global_P5 =("value", lambda x: float(x.quantile(0.05)) if x.size>0 else np.nan),
        )
    )
    # Windowed rows
    windowed = sel[(sel["ts"] >= window_start) & (sel["ts"] <= window_end)].copy().sort_values(["cmdb_id", name_col, "ts"])
    # Merge thresholds with windowed rows to preserve groups with zero window rows
    merged = thresholds.merge(windowed, on=["cmdb_id", name_col], how="left", suffixes=("_th",""))
    # Compute per-pair window aggregates
    def pair_agg(g):
        g = g.sort_values("ts")
        total_count = int(g["total_count"].iat[0])
        g_p95 = float(g["global_P95"].iat[0])
        g_p5 = float(g["global_P5"].iat[0])

        vals = g["value"]
        tss = g["ts"]

        window_count = int(vals.count())

        if window_count == 0:
            count_above = 0
            earliest_above = pd.NaT
            longest_above = 0
            count_below = 0
            earliest_below = pd.NaT
            longest_below = 0
            max_val = np.nan
            min_val = np.nan
        else:
            # masks
            above_mask = vals > g_p95
            below_mask = vals < g_p5

            count_above = int(above_mask.sum())
            earliest_above = tss[above_mask].min() if count_above>0 else pd.NaT

            # longest run helper
            cur_len = 0
            longest_above = 0
            cur_start = pd.NaT
            longest_start_above = pd.NaT
            for flag, ts in zip(above_mask.fillna(False).tolist(), tss.tolist()):
                if flag:
                    if cur_len == 0:
                        cur_start = ts
                    cur_len += 1
                else:
                    if cur_len > longest_above:
                        longest_above = cur_len
                        longest_start_above = cur_start
                    cur_len = 0
                    cur_start = pd.NaT
            if cur_len > longest_above:
                longest_above = cur_len
                longest_start_above = cur_start
            if longest_above>0:
                earliest_above = longest_start_above

            count_below = int(below_mask.sum())
            earliest_below = tss[below_mask].min() if count_below>0 else pd.NaT

            # longest below
            cur_len = 0
            longest_below = 0
            cur_start = pd.NaT
            longest_start_below = pd.NaT
            for flag, ts in zip(below_mask.fillna(False).tolist(), tss.tolist()):
                if flag:
                    if cur_len == 0:
                        cur_start = ts
                    cur_len += 1
                else:
                    if cur_len > longest_below:
                        longest_below = cur_len
                        longest_start_below = cur_start
                    cur_len = 0
                    cur_start = pd.NaT
            if cur_len > longest_below:
                longest_below = cur_len
                longest_start_below = cur_start
            if longest_below>0:
                earliest_below = longest_start_below

            max_val = float(vals.max(skipna=True)) if vals.count()>0 else np.nan
            min_val = float(vals.min(skipna=True)) if vals.count()>0 else np.nan

        ratio_above = np.nan if (g_p95 == 0) else ((max_val - g_p95) / g_p95) if not np.isnan(max_val) else np.nan
        ratio_below = np.nan if (g_p5 == 0) else ((g_p5 - min_val) / g_p5) if not np.isnan(min_val) else np.nan

        return pd.Series({
            "total_count": total_count,
            "global_P95": g_p95,
            "global_P5": g_p5,
            "window_count": window_count,
            "count_above_P95": count_above,
            "earliest_above_P95_ts": earliest_above,
            "longest_consecutive_above": int(longest_above),
            "count_below_P5": count_below,
            "earliest_below_P5_ts": earliest_below,
            "longest_consecutive_below": int(longest_below),
            "max_value_in_window": max_val,
            "min_value_in_window": min_val,
            "ratio_above": ratio_above,
            "ratio_below": ratio_below
        })

    pair_stats = merged.groupby(["cmdb_id", name_col], sort=False).apply(pair_agg).reset_index()
    return thresholds, pair_stats

# Process each source
metric_thresholds, metric_pair_stats = process_source(metric_df, "kpi_name")
trace_thresholds, trace_pair_stats = process_source(trace_df, "trace_name")
log_thresholds, log_pair_stats = process_source(log_df, "log_name")

# Helper to compute per-service aggregates from pair_stats
def service_aggregate(pair_stats, name_col):
    # pairs_with_any_breach per service
    pair_stats["breach_any"] = (pair_stats["count_above_P95"] > 0) | (pair_stats["count_below_P5"] > 0)
    # per-service aggregations
    def agg_srv(g):
        pairs_with_any_breach = int(g["breach_any"].sum())
        # max ratios: consider only finite numbers
        max_ratio_above = g["ratio_above"].replace([np.inf, -np.inf], np.nan).max()
        if pd.isna(max_ratio_above):
            max_ratio_above = np.nan
        max_ratio_below = g["ratio_below"].replace([np.inf, -np.inf], np.nan).max()
        if pd.isna(max_ratio_below):
            max_ratio_below = np.nan
        # earliest breach ts among earliest_above and earliest_below
        cand_above = g["earliest_above_P95_ts"].dropna()
        cand_below = g["earliest_below_P5_ts"].dropna()
        earliest = pd.NaT
        if not cand_above.empty or not cand_below.empty:
            all_ts = pd.concat([cand_above, cand_below])
            earliest = all_ts.min()
        return pd.Series({
            "pairs_with_any_breach": int(pairs_with_any_breach),
            "max_ratio_above": float(max_ratio_above) if not pd.isna(max_ratio_above) else np.nan,
            "max_ratio_below": float(max_ratio_below) if not pd.isna(max_ratio_below) else np.nan,
            "earliest_breach_ts": earliest
        })
    srv = pair_stats.groupby("cmdb_id", sort=False).apply(agg_srv).reset_index()
    return srv

metrics_srv = service_aggregate(metric_pair_stats, "kpi_name")
traces_srv = service_aggregate(trace_pair_stats, "trace_name")
logs_srv = service_aggregate(log_pair_stats, "log_name")

# Also need total pairs per service per source to evaluate >50% breached
def total_pairs_per_service(thresholds, name_col):
    return thresholds.groupby("cmdb_id", sort=False).size().rename("total_pairs").reset_index()

metrics_totals = total_pairs_per_service(metric_thresholds, "kpi_name")
traces_totals = total_pairs_per_service(trace_thresholds, "trace_name")
logs_totals = total_pairs_per_service(log_thresholds, "log_name")

# Merge per-service aggregates into final table (one row per service)
services_df = pd.DataFrame({"cmdb_id": targets})

final = services_df.merge(metrics_srv, on="cmdb_id", how="left", suffixes=("", "_m"))
final = final.merge(traces_srv, on="cmdb_id", how="left", suffixes=("", "_t"))
final = final.merge(logs_srv, on="cmdb_id", how="left", suffixes=("", "_l"))

# Merge totals
final = final.merge(metrics_totals.rename(columns={"size":"total_pairs"}).rename(columns={"total_pairs":"metrics_total_pairs"}), on="cmdb_id", how="left")
final = final.merge(traces_totals.rename(columns={"size":"total_pairs"}).rename(columns={"total_pairs":"traces_total_pairs"}), on="cmdb_id", how="left")
final = final.merge(logs_totals.rename(columns={"size":"total_pairs"}).rename(columns={"total_pairs":"logs_total_pairs"}), on="cmdb_id", how="left")

# The previous rename logic above created mistakes due to .size() naming; correct totals extraction properly:
# Recompute totals properly
metrics_totals = metric_thresholds.groupby("cmdb_id").size().rename("metrics_total_pairs").reset_index()
traces_totals = trace_thresholds.groupby("cmdb_id").size().rename("traces_total_pairs").reset_index()
logs_totals = log_thresholds.groupby("cmdb_id").size().rename("logs_total_pairs").reset_index()

final = services_df.merge(metrics_srv, on="cmdb_id", how="left")
final = final.merge(traces_srv, on="cmdb_id", how="left", suffixes=("_metrics", "_traces"))
final = final.merge(logs_srv, on="cmdb_id", how="left", suffixes=("", "_logs"))

final = final.merge(metrics_totals, on="cmdb_id", how="left")
final = final.merge(traces_totals, on="cmdb_id", how="left")
final = final.merge(logs_totals, on="cmdb_id", how="left")

# Rename columns to requested names and fill NaNs with sensible defaults
final = final.rename(columns={
    "pairs_with_any_breach": "metrics_pairs_with_any_breach",
    "max_ratio_above": "metrics_max_ratio_above",
    "max_ratio_below": "metrics_max_ratio_below",
    "earliest_breach_ts": "metrics_earliest_breach_ts",
    "pairs_with_any_breach_traces": "traces_pairs_with_any_breach",
    "max_ratio_above_traces": "traces_max_ratio_above",
    "max_ratio_below_traces": "traces_max_ratio_below",
    "earliest_breach_ts_traces": "traces_earliest_breach_ts",
    "pairs_with_any_breach_logs": "logs_pairs_with_any_breach",
    "max_ratio_above_logs": "logs_max_ratio_above",
    "max_ratio_below_logs": "logs_max_ratio_below",
    "earliest_breach_ts_logs": "logs_earliest_breach_ts",
})

# After merges some columns may remain with original names; map explicitly
col_map = {
    # metrics came from metrics_srv initially named pairs_with_any_breach,... after merge no suffix
    "pairs_with_any_breach": "metrics_pairs_with_any_breach",
    "max_ratio_above": "metrics_max_ratio_above",
    "max_ratio_below": "metrics_max_ratio_below",
    "earliest_breach_ts": "metrics_earliest_breach_ts",
    # traces had suffix _traces (from merge)
    "pairs_with_any_breach_traces": "traces_pairs_with_any_breach",
    "max_ratio_above_traces": "traces_max_ratio_above",
    "max_ratio_below_traces": "traces_max_ratio_below",
    "earliest_breach_ts_traces": "traces_earliest_breach_ts",
    # logs had columns without suffix due to last merge; detect if present
    "pairs_with_any_breach": "logs_pairs_with_any_breach"
}
# Apply safe renaming where columns exist
for c_old, c_new in {
    "pairs_with_any_breach": "metrics_pairs_with_any_breach",
    "max_ratio_above": "metrics_max_ratio_above",
    "max_ratio_below": "metrics_max_ratio_below",
    "earliest_breach_ts": "metrics_earliest_breach_ts",
}.items():
    if c_old in final.columns and c_new not in final.columns:
        final = final.rename(columns={c_old: c_new})

# For traces columns (they have _traces suffix after merge)
for c_old, c_new in {
    "pairs_with_any_breach_traces": "traces_pairs_with_any_breach",
    "max_ratio_above_traces": "traces_max_ratio_above",
    "max_ratio_below_traces": "traces_max_ratio_below",
    "earliest_breach_ts_traces": "traces_earliest_breach_ts",
}.items():
    if c_old in final.columns:
        final = final.rename(columns={c_old: c_new})

# For logs columns (they may be present as pairs_with_any_breach from last merge, adjust by checking)
if "pairs_with_any_breach" in final.columns and "logs_pairs_with_any_breach" not in final.columns:
    final = final.rename(columns={"pairs_with_any_breach": "logs_pairs_with_any_breach"})
if "max_ratio_above" in final.columns and "logs_max_ratio_above" not in final.columns:
    final = final.rename(columns={"max_ratio_above": "logs_max_ratio_above"})
if "max_ratio_below" in final.columns and "logs_max_ratio_below" not in final.columns:
    final = final.rename(columns={"max_ratio_below": "logs_max_ratio_below"})
if "earliest_breach_ts" in final.columns and "logs_earliest_breach_ts" not in final.columns:
    final = final.rename(columns={"earliest_breach_ts": "logs_earliest_breach_ts"})

# Ensure all expected columns exist
expected_cols = [
    "cmdb_id",
    "metrics_pairs_with_any_breach", "metrics_max_ratio_above", "metrics_max_ratio_below", "metrics_earliest_breach_ts",
    "traces_pairs_with_any_breach", "traces_max_ratio_above", "traces_max_ratio_below", "traces_earliest_breach_ts",
    "logs_pairs_with_any_breach", "logs_max_ratio_above", "logs_max_ratio_below", "logs_earliest_breach_ts",
    "metrics_total_pairs", "traces_total_pairs", "logs_total_pairs"
]
for c in expected_cols:
    if c not in final.columns:
        final[c] = np.nan

# Fill numeric NaNs with 0 for pairs_with_any_breach and totals; keep ratio NaNs as NaN
for col in ["metrics_pairs_with_any_breach", "traces_pairs_with_any_breach", "logs_pairs_with_any_breach",
            "metrics_total_pairs", "traces_total_pairs", "logs_total_pairs"]:
    final[col] = final[col].fillna(0).astype(int)

# Compute earliest breach per cell: convert to datetime or NaT
for col in ["metrics_earliest_breach_ts", "traces_earliest_breach_ts", "logs_earliest_breach_ts"]:
    final[col] = pd.to_datetime(final[col], utc=True)

# Select and order columns as requested (only keep 5 service rows)
final_table = final[[
    "cmdb_id",
    "metrics_pairs_with_any_breach", "metrics_max_ratio_above", "metrics_max_ratio_below", "metrics_earliest_breach_ts",
    "traces_pairs_with_any_breach", "traces_max_ratio_above", "traces_max_ratio_below", "traces_earliest_breach_ts",
    "logs_pairs_with_any_breach", "logs_max_ratio_above", "logs_max_ratio_below", "logs_earliest_breach_ts"
]].copy()

# 5) Summary count: services with >50% of their monitored pairs breached for any source
def breached_over_half(row):
    # metrics
    m_total = final.loc[final["cmdb_id"]==row["cmdb_id"], "metrics_total_pairs"].iat[0]
    t_total = final.loc[final["cmdb_id"]==row["cmdb_id"], "traces_total_pairs"].iat[0]
    l_total = final.loc[final["cmdb_id"]==row["cmdb_id"], "logs_total_pairs"].iat[0]
    cond = False
    if m_total>0:
        cond = cond or (row["metrics_pairs_with_any_breach"] / float(m_total) > 0.5)
    if t_total>0:
        cond = cond or (row["traces_pairs_with_any_breach"] / float(t_total) > 0.5)
    if l_total>0:
        cond = cond or (row["logs_pairs_with_any_breach"] / float(l_total) > 0.5)
    return cond

final_table["service_has_>50%_breached_any_source"] = final_table.apply(breached_over_half, axis=1)
summary_count = int(final_table["service_has_>50%_breached_any_source"].sum())

# Keep output compact: final_table (5 rows) and a small summary DataFrame
summary_df = pd.DataFrame({
    "services_with_>50%_pairs_breached_for_any_source": [summary_count],
    "total_services_evaluated": [final_table.shape[0]]
})

final_table, summary_df

```
Out[6]:
```
Summary (plain English)

- Services evaluated: ts-auth-service, ts-order-service, ts-route-service, ts-train-service, ts-travel-service (incident window 2024-01-25 11:29:40 — 11:59:40 UTC).

Per-service highlights:
- ts-auth-service
  - Metrics: 1 metric pair had any breach; max ratio_above ≈ 0.040, max ratio_below ≈ 0.575; earliest breach ≈ 2024-01-25 11:32:00 UTC.
  - Traces: 9 trace pairs breached; max ratio_above ≈ 0.288, max ratio_below ≈ 0.575; earliest trace breach 11:32:00 UTC.
  - Logs: 0 log pairs breached (no log breaches).
- ts-order-service
  - Metrics: 1 metric pair breached; metrics_max_ratio_above ≈ 0.075, max_ratio_below ≈ 0.550; earliest breach 11:32:00 UTC.
  - Traces: 26 trace pairs breached; traces_max_ratio_above ≈ 1.065 (largest relative spike among traces), max_ratio_below ≈ 0.667; earliest trace breach 11:32:00 UTC.
  - Logs: 0 log pairs breached.
- ts-route-service
  - Metrics: 1 metric pair breached; metrics_max_ratio_above ≈ 0.020, max_ratio_below ≈ 0.588; earliest breach 11:32:00 UTC.
  - Traces: 15 trace pairs breached; traces_max_ratio_above ≈ 0.142, max_ratio_below ≈ 0.604; earliest trace breach 11:32:00 UTC.
  - Logs: 0 log pairs breached.
- ts-train-service
  - Metrics: 0 metric pairs breached (no metric breaches in window).
  - Traces: 15 trace pairs breached; traces_max_ratio_above ≈ 0.413, max_ratio_below ≈ 0.604; earliest trace breach 11:32:00 UTC.
  - Logs: 0 log pairs breached.
- ts-travel-service
  - Metrics: 1 metric pair breached; metrics_max_ratio_above ≈ 0.079, max_ratio_below ≈ 0.547; earliest breach 11:32:00 UTC.
  - Traces: 36 trace pairs breached (largest absolute count); traces_max_ratio_above ≈ 0.826, max_ratio_below ≈ 0.559; earliest trace breach 11:32:00 UTC.
  - Logs: 0 log pairs breached.

Overall summary:
- Number of services where more than 50% of monitored pairs were breached for at least one source: 5 out of 5 services.
- Notes:
  - Trace telemetry shows the largest number of breached pairs and the largest relative spikes (notably ts-order and ts-travel).
  - Metrics show only a few breached metric pairs per service (typically 0–1), with generally small ratio_above values except some moderate memory/disk/latency excursions noted earlier.
  - Logs show no breaches in the evaluated log_name pairs for these services.

The original code execution output of IPython Kernel is also provided below for reference:

(             cmdb_id  metrics_pairs_with_any_breach  metrics_max_ratio_above  metrics_max_ratio_below metrics_earliest_breach_ts  traces_pairs_with_any_breach  traces_max_ratio_above  traces_max_ratio_below traces_earliest_breach_ts  logs_pairs_with_any_breach  logs_max_ratio_above  logs_max_ratio_below logs_earliest_breach_ts  service_has_>50%_breached_any_source
0    ts-auth-service                              1                 0.040312                 0.575290  2024-01-25 11:32:00+00:00                             9                0.288387                0.575290 2024-01-25 11:32:00+00:00                           0                   NaN                   NaN                     NaT                                  True
1   ts-order-service                              1                 0.074610                 0.550321  2024-01-25 11:32:00+00:00                            26                1.065480                0.666667 2024-01-25 11:32:00+00:00                           0                   NaN                   NaN                     NaT                                  True
2   ts-route-service                              1                 0.020209                 0.588385  2024-01-25 11:32:00+00:00                            15                0.141969                0.603960 2024-01-25 11:32:00+00:00                           0                   NaN                   NaN                     NaT                                  True
3   ts-train-service                              0                      NaN                      NaN                        NaT                            15                0.413213                0.603960 2024-01-25 11:32:00+00:00                           0                   NaN                   NaN                     NaT                                  True
4  ts-travel-service                              1                 0.079477                 0.546507  2024-01-25 11:32:00+00:00                            36                0.826283                0.559342 2024-01-25 11:32:00+00:00                           0                   NaN                   NaN                     NaT                                  True,    services_with_>50%_pairs_breached_for_any_source  total_services_evaluated
0                                                 5                         5)```
```